# 🩺 GlucoAI — Glucose Monitoring & Diabetes Risk Analysis

**Production-grade ML notebook for clinical glucose intelligence**

---
### Pipeline Overview
1. Data loading & preprocessing
2. Exploratory Data Analysis (EDA)
3. Feature engineering
4. Model training (Random Forest + Logistic Regression baseline)
5. Model evaluation (ROC-AUC, CV, confusion matrix)
6. SHAP explainability
7. Anomaly detection (Isolation Forest)
8. Glucose trend analysis
9. Meal recommendation system
10. End-to-end inference example

---
_Author: GlucoAI Team | Version: 1.0.0 | Framework: scikit-learn, SHAP, Plotly_

In [ ]:
# ── Standard imports ──
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

# ── ML imports ──
import shap
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report,
    confusion_matrix, precision_recall_curve, average_precision_score
)
from sklearn.pipeline import Pipeline

# ── Style ──
plt.style.use('dark_background')
PALETTE = ['#00C9A7', '#FF4757', '#FFA502', '#2ED573', '#A8EDEA']
sns.set_palette(PALETTE)

print('✅ Imports successful')
print(f'NumPy {np.__version__} | Pandas {pd.__version__} | SHAP {shap.__version__}')

## 1. Data Generation & Loading

In [ ]:
def generate_clinical_dataset(n_samples: int = 3000, seed: int = 42) -> pd.DataFrame:
    """
    Generate a realistic synthetic diabetes dataset.
    Feature distributions derived from NHANES and Pima benchmark.
    """
    rng = np.random.default_rng(seed)

    age            = rng.integers(20, 80, n_samples).astype(float)
    bmi            = rng.normal(27, 5, n_samples).clip(15, 50)
    glucose        = rng.normal(120, 30, n_samples).clip(60, 300)
    hba1c          = rng.normal(6.0, 1.2, n_samples).clip(4.0, 12.0)
    blood_pressure = rng.normal(75, 12, n_samples).clip(50, 120)
    insulin        = rng.exponential(80, n_samples).clip(0, 400)
    skin_thickness = rng.normal(25, 10, n_samples).clip(5, 60)
    pregnancies    = rng.integers(0, 10, n_samples).astype(float)
    activity_level = rng.integers(0, 5, n_samples).astype(float)  # 0=sedentary..4=very_active
    sleep_hours    = rng.normal(7, 1.2, n_samples).clip(4, 10)
    stress_level   = rng.integers(0, 3, n_samples).astype(float)  # 0=low,1=mod,2=high
    family_history = rng.integers(0, 2, n_samples).astype(float)
    smoker         = rng.integers(0, 2, n_samples).astype(float)
    hypertension   = rng.integers(0, 2, n_samples).astype(float)

    # Weighted clinical risk model
    log_odds = (
        -8.0
        + 0.04 * age
        + 0.10 * bmi
        + 0.02 * glucose
        + 0.50 * hba1c
        + 0.005 * blood_pressure
        + 0.003 * insulin
        + 0.60 * family_history
        + 0.40 * hypertension
        - 0.25 * activity_level
        - 0.15 * sleep_hours
        + 0.20 * stress_level
        + 0.30 * smoker
    )
    prob = (1 / (1 + np.exp(-log_odds)) + rng.normal(0, 0.05, n_samples)).clip(0, 1)
    is_diabetic = (prob > 0.5).astype(int)

    return pd.DataFrame({
        'age': age, 'bmi': bmi, 'glucose': glucose, 'hba1c': hba1c,
        'blood_pressure': blood_pressure, 'insulin': insulin,
        'skin_thickness': skin_thickness, 'pregnancies': pregnancies,
        'activity_level': activity_level, 'sleep_hours': sleep_hours,
        'stress_level': stress_level, 'family_history': family_history,
        'smoker': smoker, 'hypertension': hypertension,
        'is_diabetic': is_diabetic,
    })

df = generate_clinical_dataset(n_samples=3000)
print(f'Dataset shape: {df.shape}')
print(f'Class balance: {df.is_diabetic.value_counts().to_dict()}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# ── 2a. Statistical summary ──
print('=== Statistical Summary ===')
df.describe().round(2)

In [ ]:
# ── 2b. Feature distributions by class ──
FEATURES = ['glucose', 'hba1c', 'bmi', 'age', 'blood_pressure', 'insulin']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, feat in zip(axes, FEATURES):
    for label, color, name in [(0, '#00C9A7', 'Non-Diabetic'), (1, '#FF4757', 'Diabetic')]:
        subset = df[df.is_diabetic == label][feat]
        ax.hist(subset, bins=40, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_facecolor('#0A0E1A')
    ax.tick_params(colors='#8B9AB5')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333')

plt.suptitle('Feature Distributions by Diabetes Status', fontsize=14, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight', facecolor='#080D18')
plt.show()

In [ ]:
# ── 2c. Correlation heatmap ──
FEATURE_COLS = [
    'age', 'bmi', 'glucose', 'hba1c', 'blood_pressure', 'insulin',
    'skin_thickness', 'pregnancies', 'activity_level', 'sleep_hours',
    'stress_level', 'family_history', 'smoker', 'hypertension', 'is_diabetic'
]
corr = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
cmap = sns.diverging_palette(145, 10, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, center=0, linewidths=0.5,
            annot=True, fmt='.2f', annot_kws={'size': 8}, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocessing Pipeline

In [ ]:
FEATURE_COLS = [
    'age', 'bmi', 'glucose', 'hba1c', 'blood_pressure', 'insulin',
    'skin_thickness', 'pregnancies', 'activity_level', 'sleep_hours',
    'stress_level', 'family_history', 'smoker', 'hypertension'
]

DISPLAY_NAMES = {
    'age': 'Age', 'bmi': 'BMI', 'glucose': 'Glucose',
    'hba1c': 'HbA1c', 'blood_pressure': 'Blood Pressure',
    'insulin': 'Insulin', 'skin_thickness': 'Skin Thickness',
    'pregnancies': 'Pregnancies', 'activity_level': 'Activity Level',
    'sleep_hours': 'Sleep Hours', 'stress_level': 'Stress Level',
    'family_history': 'Family History', 'smoker': 'Smoker',
    'hypertension': 'Hypertension'
}

def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """Clinical data preprocessing: imputation + clipping."""
    df = df.copy()
    for col in FEATURE_COLS:
        df[col] = df[col].fillna(df[col].median())
    # Physiological bounds
    clip_bounds = {
        'bmi': (10, 60), 'glucose': (40, 400), 'hba1c': (3.5, 15.0),
        'age': (0, 110), 'blood_pressure': (40, 130), 'insulin': (0, 600),
        'sleep_hours': (2, 12)
    }
    for col, (lo, hi) in clip_bounds.items():
        if col in df.columns:
            df[col] = df[col].clip(lo, hi)
    return df

df_clean = preprocess(df)
X = df_clean[FEATURE_COLS].values
y = df_clean['is_diabetic'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')
print(f'Class distribution train: {np.bincount(y_train)}')
print(f'Class distribution test:  {np.bincount(y_test)}')

## 4. Model Training

In [ ]:
# ── 4a. Logistic Regression baseline ──
lr = LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train_s, y_train)
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test_s)[:, 1])
print(f'Logistic Regression ROC-AUC: {lr_auc:.4f}')

In [ ]:
# ── 4b. Random Forest (primary model) ──
rf = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_split=10,
    min_samples_leaf=5, max_features='sqrt',
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train_s, y_train)
rf_prob = rf.predict_proba(X_test_s)[:, 1]
rf_auc = roc_auc_score(y_test, rf_prob)
print(f'Random Forest ROC-AUC: {rf_auc:.4f}')

# Cross-validation
cv_scores = cross_val_score(rf, X_train_s, y_train,
                            cv=StratifiedKFold(n_splits=5), scoring='roc_auc')
print(f'CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

print('\n=== Classification Report ===')
print(classification_report(y_test, rf.predict(X_test_s), target_names=['Non-Diabetic', 'Diabetic']))

## 5. Model Evaluation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── ROC Curve ──
ax = axes[0]
for model, name, color in [(lr, 'Logistic Regression', '#FFA502'), (rf, 'Random Forest', '#00C9A7')]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test_s)[:, 1])
    auc = roc_auc_score(y_test, model.predict_proba(X_test_s)[:, 1])
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold'); ax.legend(); ax.set_facecolor('#0A0E1A')
ax.grid(True, alpha=0.1)

# ── Precision-Recall ──
ax = axes[1]
for model, name, color in [(lr, 'LR', '#FFA502'), (rf, 'RF', '#00C9A7')]:
    prec, rec, _ = precision_recall_curve(y_test, model.predict_proba(X_test_s)[:, 1])
    ap = average_precision_score(y_test, model.predict_proba(X_test_s)[:, 1])
    ax.plot(rec, prec, label=f'{name} (AP={ap:.3f})', color=color, lw=2)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', fontweight='bold'); ax.legend(); ax.set_facecolor('#0A0E1A')
ax.grid(True, alpha=0.1)

# ── Confusion Matrix ──
ax = axes[2]
cm = confusion_matrix(y_test, rf.predict(X_test_s))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['Predicted: No', 'Predicted: Yes'],
            yticklabels=['Actual: No', 'Actual: Yes'], ax=ax)
ax.set_title('Random Forest Confusion Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight', facecolor='#080D18')
plt.show()

## 6. SHAP Explainability

In [ ]:
# ── 6a. Compute SHAP values ──
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test_s)

# For binary RF, shap_values[1] = positive class
sv_positive = shap_values[1] if isinstance(shap_values, list) else shap_values
display_names = [DISPLAY_NAMES.get(f, f) for f in FEATURE_COLS]

print('SHAP values computed for', len(X_test_s), 'test samples')
print('Mean |SHAP| per feature:')
mean_shap = np.abs(sv_positive).mean(axis=0)
for name, val in sorted(zip(display_names, mean_shap), key=lambda x: -x[1]):
    bar = '█' * int(val * 200)
    print(f'  {name:<22} {val:.4f}  {bar}')

In [ ]:
# ── 6b. SHAP Summary Plot ──
plt.figure(figsize=(10, 7))
shap.summary_plot(
    sv_positive, X_test_s,
    feature_names=display_names,
    plot_type='dot',
    show=False,
    max_display=14,
)
plt.title('SHAP Feature Impact (Positive = Increases Diabetes Risk)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6c. Waterfall for single patient ──
PATIENT_IDX = 5  # index in test set
exp = shap.Explanation(
    values=sv_positive[PATIENT_IDX],
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    data=X_test_s[PATIENT_IDX],
    feature_names=display_names
)
shap.plots.waterfall(exp, show=True)

## 7. Anomaly Detection

In [ ]:
def detect_glucose_anomalies(
    readings: list,
    contamination: float = 0.05
) -> pd.DataFrame:
    """
    Isolation Forest anomaly detection on a glucose time-series.
    Augments with clinical threshold rules.
    """
    arr = np.array(readings)
    n = len(arr)
    window = 3

    # Build sliding-window feature matrix
    feats = []
    for i in range(n):
        w = arr[max(0, i-window): i+1]
        feats.append([arr[i], w.mean(), w.std() if len(w) > 1 else 0, arr[i] - arr[i-1] if i > 0 else 0])

    F = np.array(feats)
    F_s = StandardScaler().fit_transform(F)

    iso = IsolationForest(contamination=contamination, n_estimators=100, random_state=42)
    preds = iso.fit_predict(F_s)
    scores = iso.score_samples(F_s)

    df = pd.DataFrame({
        'index': range(n),
        'glucose': readings,
        'ml_anomaly': preds == -1,
        'anomaly_score': scores,
        'is_hypo': arr < 70,
        'is_hyper': arr > 180,
        'is_critical': (arr < 54) | (arr > 250),
    })
    df['is_anomaly'] = df['ml_anomaly'] | df['is_hypo'] | df['is_hyper']
    return df


# ── Demo glucose series ──
demo_readings = [
    95, 148, 110, 165, 130, 158, 105, 92, 145, 108,
    70, 168, 128, 310,  # spike!
    125, 160, 99, 45,   # critical low!
    142, 115, 170
]

anomaly_df = detect_glucose_anomalies(demo_readings)
print(f'Total readings: {len(demo_readings)}')
print(f'Anomalies detected: {anomaly_df.is_anomaly.sum()}')
print(f'Critical events: {anomaly_df.is_critical.sum()}')
anomaly_df[anomaly_df.is_anomaly]

In [ ]:
# ── Anomaly visualization ──
fig = go.Figure()

# Target range
fig.add_hrect(y0=70, y1=180, fillcolor='rgba(0,201,167,0.08)', line_width=0)

# Main line
fig.add_trace(go.Scatter(
    x=anomaly_df.index, y=anomaly_df.glucose,
    mode='lines+markers',
    line=dict(color='#00C9A7', width=2),
    marker=dict(
        color=['#FF4757' if a else '#00C9A7' for a in anomaly_df.is_anomaly],
        size=[14 if a else 7 for a in anomaly_df.is_anomaly],
        symbol=['diamond' if a else 'circle' for a in anomaly_df.is_anomaly]
    ),
    name='Glucose Readings'
))

for y, color, label in [(70, '#FF475780', 'Low (70)'), (180, '#FFA50280', 'High (180)'), (250, '#FF475740', 'Critical (250)')]:
    fig.add_hline(y=y, line_dash='dash', line_color=color, annotation_text=label)

fig.update_layout(
    title='Glucose Anomaly Detection', height=380,
    paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
    font_color='#8B9AB5',
    xaxis=dict(gridcolor='rgba(255,255,255,0.05)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.05)', title='Glucose (mg/dL)'),
)
fig.show()

## 8. Trend Analysis

In [ ]:
def compute_trend_stats(readings: list, window: int = 4) -> pd.DataFrame:
    """Rolling statistics for glucose trend analysis."""
    df = pd.DataFrame({'glucose': readings})
    df['rolling_mean'] = df['glucose'].rolling(window, min_periods=1).mean()
    df['rolling_std']  = df['glucose'].rolling(window, min_periods=1).std().fillna(0)
    df['delta']        = df['glucose'].diff().fillna(0)
    df['is_spike']     = df['delta'].abs() > 50
    return df


def time_in_range(readings: list) -> dict:
    """ADA Time-in-Range metrics (70–180 mg/dL target)."""
    arr = np.array(readings)
    n = len(arr)
    return {
        'in_range_pct':    round(((arr >= 70) & (arr <= 180)).sum() / n * 100, 1),
        'below_range_pct': round((arr < 70).sum() / n * 100, 1),
        'above_range_pct': round((arr > 180).sum() / n * 100, 1),
        'mean': round(arr.mean(), 1),
        'std':  round(arr.std(), 1),
        'cv':   round(arr.std() / arr.mean() * 100, 1),
        'estimated_hba1c': round((arr.mean() + 46.7) / 28.7, 1),
    }


def trend_direction(readings: list, window: int = 7) -> str:
    """Linear regression slope to determine trend."""
    r = readings[-window:] if len(readings) >= window else readings
    slope = np.polyfit(range(len(r)), r, 1)[0]
    return 'worsening' if slope > 2 else 'improving' if slope < -2 else 'stable'


trend_df = compute_trend_stats(demo_readings)
tir = time_in_range(demo_readings)
direction = trend_direction(demo_readings)

print('=== Time in Range ===')
for k, v in tir.items():
    print(f'  {k:<25} {v}')
print(f'\nTrend direction: {direction.upper()}')
print(f'Spike count: {trend_df.is_spike.sum()}')

In [ ]:
# ── Rolling trend chart ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Target zone
ax1.axhspan(70, 180, alpha=0.08, color='#00C9A7', label='Target Range')

# Raw readings
ax1.scatter(trend_df.index, trend_df.glucose,
           c=['#FF4757' if a else '#00C9A730' for a in anomaly_df.is_anomaly],
           s=80, zorder=3)

# Rolling mean + band
ax1.plot(trend_df.index, trend_df.rolling_mean, color='#00C9A7', lw=2.5, label='Rolling Mean')
ax1.fill_between(
    trend_df.index,
    trend_df.rolling_mean - trend_df.rolling_std,
    trend_df.rolling_mean + trend_df.rolling_std,
    alpha=0.15, color='#00C9A7', label='±1 SD'
)

# Spikes
spike_idx = trend_df[trend_df.is_spike].index
spike_vals = trend_df.loc[spike_idx, 'glucose']
ax1.scatter(spike_idx, spike_vals, marker='^', color='#FFA502', s=120, zorder=4, label=f'Spikes ({len(spike_idx)})')

ax1.axhline(70, color='#FF4757', lw=1, ls='--', alpha=0.6)
ax1.axhline(180, color='#FFA502', lw=1, ls='--', alpha=0.6)
ax1.set_ylabel('Glucose (mg/dL)'); ax1.legend(); ax1.set_facecolor('#0A0E1A')
ax1.set_title('Glucose Trend Analysis', fontsize=13, fontweight='bold')

# Delta / velocity
ax2.bar(trend_df.index, trend_df.delta,
       color=['#FF4757' if d > 50 else '#2ED573' if d < -50 else '#00C9A740' for d in trend_df.delta])
ax2.axhline(0, color='white', lw=0.5, alpha=0.3)
ax2.set_ylabel('Δ Glucose'); ax2.set_xlabel('Reading Index')
ax2.set_facecolor('#0A0E1A')

plt.tight_layout()
plt.savefig('trend_analysis.png', dpi=150, bbox_inches='tight', facecolor='#080D18')
plt.show()

## 9. Meal Recommendation System

In [ ]:
import os

GI_PATH = '../data/glycemic_index.csv'
if os.path.exists(GI_PATH):
    gi_db = pd.read_csv(GI_PATH)
else:
    # Minimal inline fallback
    gi_db = pd.DataFrame({
        'food_name': ['White rice', 'Brown rice', 'White bread', 'Whole grain bread', 'Apple', 'Banana',
                      'Oatmeal', 'Lentils', 'Salmon', 'Broccoli', 'Potato', 'Chips'],
        'glycemic_index': [72, 50, 75, 51, 36, 51, 55, 32, 0, 10, 82, 54],
        'diabetes_friendly': [False, True, False, True, True, True, True, True, True, True, False, False],
        'alternatives': ['Brown rice', '', 'Whole grain bread', '', '', 'Berries', '',
                         '', '', '', 'Cauliflower mash', 'Nuts'],
        'notes': ['High GI', 'Good choice', 'Avoid', 'Good fiber', 'Low GI', 'Moderate',
                  'Slow release', 'Excellent', 'Omega-3', 'Excellent', 'High GI', 'Avoid'],
    })


def analyze_meal(meal_text: str, gi_db: pd.DataFrame) -> dict:
    """Analyze a meal description against the GI database."""
    meal_lower = meal_text.lower()
    matched = gi_db[
        gi_db['food_name'].str.lower().apply(
            lambda name: any(w in meal_lower for w in name.split())
        )
    ]

    if matched.empty:
        return {'status': 'no_match', 'message': 'Food items not found in database'}

    avg_gi = matched['glycemic_index'].mean()
    problematic = matched[matched['diabetes_friendly'] == False]
    safe = matched[matched['diabetes_friendly'] == True]

    alternatives = []
    for _, row in problematic.iterrows():
        alts = str(row.get('alternatives', ''))
        if alts and alts != 'nan':
            alternatives.extend(a.strip() for a in alts.split(','))

    return {
        'matched_foods': matched['food_name'].tolist(),
        'avg_gi': round(avg_gi, 1),
        'impact': 'low' if avg_gi < 55 else 'moderate' if avg_gi < 70 else 'high',
        'rating': 'good' if avg_gi < 50 else 'moderate' if avg_gi < 65 else 'poor',
        'avoid': problematic['food_name'].tolist(),
        'alternatives': list(set(alternatives))[:4],
        'safe_foods': safe['food_name'].tolist(),
    }


# ── Test meals ──
test_meals = [
    'white rice with fried chicken and potato chips',
    'oatmeal with apple and almonds',
    'burger with french fries and soda',
    'salmon with broccoli and lentils',
]

for meal in test_meals:
    result = analyze_meal(meal, gi_db)
    print(f'\n🍽️  {meal}')
    print(f'   GI: {result.get("avg_gi", "N/A")} | Impact: {result.get("impact","?")} | Rating: {result.get("rating","?")}')
    if result.get('avoid'):
        print(f'   ❌ Avoid: {", ".join(result["avoid"])}')
    if result.get('alternatives'):
        print(f'   ✅ Try instead: {", ".join(result["alternatives"])}')

## 10. End-to-End Inference

In [ ]:
# ── Full inference pipeline ──

sample_patient = {
    'age': 55,
    'bmi': 31.2,
    'glucose': 165.0,
    'hba1c': 7.1,
    'blood_pressure': 85.0,
    'insulin': 120.0,
    'skin_thickness': 30.0,
    'pregnancies': 2.0,
    'activity_level': 1.0,  # light
    'sleep_hours': 5.5,
    'stress_level': 2.0,   # high
    'family_history': 1.0,
    'smoker': 0.0,
    'hypertension': 1.0,
}

# Build feature vector
x = np.array([[sample_patient[f] for f in FEATURE_COLS]])
x_scaled = scaler.transform(x)

# Predict
prob = rf.predict_proba(x_scaled)[0, 1]
risk_level = 'HIGH' if prob >= 0.65 else 'MEDIUM' if prob >= 0.35 else 'LOW'
risk_color = '🔴' if risk_level == 'HIGH' else '🟡' if risk_level == 'MEDIUM' else '🟢'

# SHAP for this patient
sv = explainer.shap_values(x_scaled)
patient_shap = sv[1][0] if isinstance(sv, list) else sv[0]
top_factors = sorted(zip(display_names, patient_shap), key=lambda t: abs(t[1]), reverse=True)[:5]

print('═' * 50)
print('  GLUCOAI — Patient Risk Report')
print('═' * 50)
print(f'  Risk Score: {prob:.1%}')
print(f'  Risk Level: {risk_color} {risk_level}')
print()
print('  Top Contributing Factors:')
for name, shap_val in top_factors:
    direction = '↑ increases risk' if shap_val > 0 else '↓ decreases risk'
    bar_char = '█' if shap_val > 0 else '░'
    bar = bar_char * min(20, int(abs(shap_val) * 100))
    print(f'  {name:<22} {shap_val:+.3f}  {bar}  {direction}')
print('═' * 50)

In [ ]:
# ── Save trained artifacts ──
import pickle
from pathlib import Path

MODEL_DIR = Path('../app/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

artifacts = {
    'model': rf,
    'scaler': scaler,
    'explainer': explainer,
    'feature_importances': dict(zip(FEATURE_COLS, rf.feature_importances_.tolist())),
    'version': '1.0.0',
}

with open(MODEL_DIR / 'risk_model.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print(f'✅ Model saved to {MODEL_DIR / "risk_model.pkl"}')
print(f'   File size: {(MODEL_DIR / "risk_model.pkl").stat().st_size / 1024:.1f} KB')

---
## ✅ Summary

| Component | Status | Notes |
|-----------|--------|-------|
| Dataset generation | ✅ | 3,000 synthetic patients, clinical distributions |
| Preprocessing | ✅ | Imputation, physiological clipping |
| EDA | ✅ | Distributions, correlations |
| Logistic Regression | ✅ | Baseline model |
| Random Forest | ✅ | Primary model, class-weighted |
| Cross-Validation | ✅ | 5-fold Stratified K-Fold |
| SHAP Explainability | ✅ | Global + per-patient waterfall |
| Anomaly Detection | ✅ | Isolation Forest + clinical thresholds |
| Trend Analysis | ✅ | Rolling stats, TIR, spike detection |
| Meal Recommendations | ✅ | Glycemic index database |
| Inference Pipeline | ✅ | End-to-end single patient prediction |
| Model Persistence | ✅ | Saved to `app/models/risk_model.pkl` |